# Lab 14: Graph Neural Networks from an Explicit Adjacency Matrix

            **Duration:** 3 hours  
            **Lecture alignment:** Week 14 — Graph neural networks  
            **CLO mapping:** CLO-1, CLO-2, CLO-3  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Represent an undirected graph with explicit adjacency and feature matrices.
- Implement symmetric normalization and two GCN layers using native PyTorch.
- Compare graph-aware message passing with an edge-ignorant baseline.

            ## Three-hour activity plan

            - 0–30 min: graph representation and node/graph task distinction
- 30–70 min: self-loops and normalized adjacency
- 70–125 min: implement/train GCN
- 125–155 min: MLP baseline and visualization
- 155–180 min: GraphSAGE/GAT comparison and checks


## Book grounding

            - Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.
- Bishop and Bishop, *Deep Learning: Foundations and Concepts*, Springer, 2024.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20274
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_14")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_14"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 14, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Will a GCN classify unlabeled nodes better than an MLP that ignores edges? Identify how graph homophily affects that prediction.

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Explicit graph tensors and normalized message passing


In [ ]:
A=torch.tensor([
    [0,1,1,0,0,0,0,0,0,0],
    [1,0,1,1,0,0,0,0,0,0],
    [1,1,0,1,0,0,0,0,0,0],
    [0,1,1,0,1,0,0,0,0,0],
    [0,0,0,1,0,1,0,0,0,0],
    [0,0,0,0,1,0,1,0,0,0],
    [0,0,0,0,0,1,0,1,1,0],
    [0,0,0,0,0,0,1,0,1,1],
    [0,0,0,0,0,0,1,1,0,1],
    [0,0,0,0,0,0,0,1,1,0],
],dtype=torch.float32)
labels=torch.tensor([0,0,0,0,0,1,1,1,1,1]);X=torch.eye(10)
I=torch.eye(len(A));degree=(A+I).sum(1);D_inv_sqrt=torch.diag(degree.pow(-.5));A_norm=D_inv_sqrt@(A+I)@D_inv_sqrt
train_mask=torch.tensor([True,False,True,False,False,True,False,True,False,False]);test_mask=~train_mask
assert torch.allclose(A,A.T) and torch.allclose(A_norm,A_norm.T)
print({"nodes":len(A),"undirected_edges":int(A.sum().item()/2),"features":tuple(X.shape)})


## Activity 2 — Native-PyTorch GCN and edge-ignorant MLP


In [ ]:
class GCNLayer(nn.Module):
    def __init__(self,in_features,out_features):super().__init__();self.linear=nn.Linear(in_features,out_features,bias=False)
    def forward(self,x,adj):return adj@self.linear(x)
class GCN(nn.Module):
    def __init__(self):super().__init__();self.g1=GCNLayer(10,8);self.g2=GCNLayer(8,2)
    def forward(self,x,adj,return_embedding=False):
        h=F.relu(self.g1(x,adj));out=self.g2(h,adj);return (out,h) if return_embedding else out
class NodeMLP(nn.Module):
    def __init__(self):super().__init__();self.net=nn.Sequential(nn.Linear(10,8),nn.ReLU(),nn.Linear(8,2))
    def forward(self,x,adj=None):return self.net(x)
def train_graph(model,use_graph):
    model=model.to(DEVICE);x=X.to(DEVICE);adj=A_norm.to(DEVICE);y=labels.to(DEVICE);train_index=train_mask.to(DEVICE);opt=torch.optim.Adam(model.parameters(),lr=.03,weight_decay=1e-3);hist=[]
    for _ in range(180 if FAST_MODE else 500):
        model.train();opt.zero_grad();logits=model(x,adj) if use_graph else model(x);loss=F.cross_entropy(logits[train_index],y[train_index]);loss.backward();opt.step();hist.append(loss.item())
    model.eval();
    with torch.no_grad():logits=model(x,adj) if use_graph else model(x);pred=logits.argmax(1).cpu();test_acc=(pred[test_mask]==labels[test_mask]).float().mean().item()
    return model,hist,test_acc,pred
gcn,gcn_hist,gcn_acc,gcn_pred=train_graph(GCN(),True);mlp,mlp_hist,mlp_acc,mlp_pred=train_graph(NodeMLP(),False)
with torch.no_grad():_,embedding=gcn(X.to(DEVICE),A_norm.to(DEVICE),True);embedding=embedding.cpu()
print({"GCN_test_accuracy":gcn_acc,"MLP_test_accuracy":mlp_acc})


## Activity 3 — Visualize message-passing predictions


In [ ]:
coords=torch.tensor([[-2,1],[-1.5,2],[-1,1],[-.5,2],[0,1],[1,1],[1.5,2],[2,1],[2.5,2],[3,1]],dtype=torch.float32)
fig,axes=plt.subplots(1,3,figsize=(12,3.6))
for i,j in torch.nonzero(torch.triu(A),as_tuple=False):
    for ax in axes:ax.plot([coords[i,0],coords[j,0]],[coords[i,1],coords[j,1]],color="lightgray",zorder=0)
for ax,values,title in zip(axes,[labels,gcn_pred,mlp_pred],["True communities","GCN predictions","MLP predictions"]):
    ax.scatter(coords[:,0],coords[:,1],c=values,cmap="coolwarm",s=100,edgecolor="black");ax.set_title(title);ax.axis("off")
fig.tight_layout();fig.savefig(ARTIFACT_DIR/"gcn_predictions.png",dpi=150);plt.show()
torch.save(gcn.state_dict(),ARTIFACT_DIR/"gcn.pt")


## Automated checks


In [ ]:
assert A.shape==(10,10) and X.shape==(10,10) and embedding.shape==(10,8)
assert torch.isfinite(A_norm).all() and gcn_hist[-1]<gcn_hist[0]
assert 0<=gcn_acc<=1 and set(gcn_pred.tolist())<={0,1}
assert (ARTIFACT_DIR/"gcn.pt").exists()
print("All Lab 14 checks passed.")


## Deliverables

                - Explicit adjacency and normalization checks
- Native-PyTorch GCN implementation - no torch-geometric
- GCN/MLP comparison and graph visualization

                Submit the executed notebook and the files created in `/content/artifacts/lab_14/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    neighbor_mean=A/(A.sum(1,keepdim=True)+1e-8)@X
    print("Mean-aggregated GraphSAGE-style feature shape:",neighbor_mean.shape)
else:
    print("Extension disabled: mean GraphSAGE or a single-head graph-attention layer.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
